# 📘 Guide 10: Intermediate Python: type hints and Pydantic

**Who it's for:** everyone, before week 1 day 3. GradeWise is built on **Pydantic** models: they describe exam papers, answers and grades, and they're how we get reliable structured output from LLMs. **Time:** about 30 minutes.

### 🎯 You'll learn
1. Type hints, and why they matter even though Python doesn't enforce them
2. Pydantic models: validation, defaults and constraints
3. Nesting models, and turning them into (and out of) JSON
4. How this connects to LLM structured output
5. pydantic-settings, the engine behind `gradewise/config.py`

## Part 1: Type hints

A type hint says what a value *should* be. Python itself ignores them at runtime, but your editor, your AI assistant, linters and Pydantic all use them.

In [ ]:
def total(marks: list[float], bonus: float = 0.0) -> float:
    return sum(marks) + bonus


print(total([4, 3.5, 2]))
# 💡 total(["4", "3"]) is "wrong" by the hints, yet Python only notices when sum() crashes at runtime

# Common hint shapes you'll see in the course
student_marks: dict[str, float] = {"Sam": 9.5}
comment: str | None = None  # "a string, or None"

## Part 2: Your first Pydantic model

A Pydantic model is a class where the type hints **are enforced**. It checks and converts data when you create it. Here's a simplified version of the course's `ExamPaper`:

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Question(BaseModel):
    number: str = Field(description="Question label, e.g. 'Q1'")
    text: str
    max_marks: float = Field(gt=0, le=20)


q = Question(number="Q1", text="Define a stack.", max_marks="5")  # 💡 "5" is converted to 5.0
print(q)
print(type(q.max_marks).__name__)

try:
    Question(number="Q2", text="Explain recursion.", max_marks=-3)
except ValidationError as e:
    print("\n❌ Rejected:\n", e)

## Part 3: Nesting, computed values and JSON

In [ ]:
from pydantic import computed_field


class ExamPaper(BaseModel):
    course: str = "CS201"
    title: str
    questions: list[Question]

    @computed_field
    @property
    def total_marks(self) -> float:
        return sum(q.max_marks for q in self.questions)


paper = ExamPaper(
    title="Data Structures Midterm",
    questions=[
        {"number": "Q1", "text": "Define a stack.", "max_marks": 5},  # 💡 dicts become Question objects
        {"number": "Q2", "text": "Trace binary search on [1, 4, 9].", "max_marks": 10},
    ],
)
print(paper.total_marks)

as_json = paper.model_dump_json(indent=2)
print(as_json)

again = ExamPaper.model_validate_json(as_json)
print("Round trip equal:", again == paper)

## Part 4: Why LLMs love Pydantic

A model can describe itself as a **JSON Schema**. LangChain sends that schema to the LLM, and the LLM replies with JSON that Pydantic validates. That's how you'll get marks back as `GradeResult` objects instead of free text:

In [ ]:
import json

schema = Question.model_json_schema()
print(json.dumps(schema, indent=2)[:600])

In the labs (needs an API key, so shown here as a snippet):

```python
from gradewise.models import chat_model

llm = chat_model().with_structured_output(Question)
q = llm.invoke("Write one short CS201 exam question about queues, worth 4 marks.")
print(q.number, q.max_marks)      # a real, validated Question object
```

The **real** course schemas live in `gradewise/schemas/` (`ExamPaper`, `AnswerSheet`, `GradeResult`, `ReportCard`). Have a look: they use exactly what you just learned.

In [ ]:
from gradewise.schemas.exam import ExamPaper as CourseExamPaper

print("The course's ExamPaper fields:", list(CourseExamPaper.model_fields))

## Part 5: pydantic-settings

`BaseSettings` is a Pydantic model that fills itself from environment variables and `.env`, converting `"true"` to `True` along the way. That's all `gradewise/config.py` is:

In [ ]:
import os

from pydantic_settings import BaseSettings


class DemoSettings(BaseSettings):
    demo_budget_mode: bool = False
    demo_max_students: int = 30


os.environ["DEMO_BUDGET_MODE"] = "true"
os.environ["DEMO_MAX_STUDENTS"] = "120"
s = DemoSettings()
print(s, "| types:", type(s.demo_budget_mode).__name__, type(s.demo_max_students).__name__)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Add a <code>difficulty</code> field to <code>Question</code> that only accepts <code>"easy"</code>, <code>"medium"</code> or <code>"hard"</code> (hint: <code>typing.Literal</code>). Then write a <code>Grade</code> model with <code>marks</code> that must be ≥ 0, and a <code>model_validator</code> that rejects marks above the question's <code>max_marks</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Pydantic documentation: <a href="https://docs.pydantic.dev/latest/">docs.pydantic.dev/latest/</a><br/>• pydantic-settings: <a href="https://docs.pydantic.dev/latest/concepts/pydantic_settings/">docs.pydantic.dev/latest/concepts/pydantic_settings/</a><br/>• Python typing cheat sheet (mypy): <a href="https://mypy.readthedocs.io/en/stable/cheat_sheet_py3.html">mypy.readthedocs.io/en/stable/cheat_sheet_py3.html</a></span>
        </td>
    </tr>
</table>